# CIFAR-10 Image Classification with AutoGluon

Notebook ini menampilkan pipeline lengkap untuk melakukan klasifikasi gambar pada dataset CIFAR-10 menggunakan [AutoGluon](https://auto.gluon.ai/stable/index.html). Notebook disiapkan untuk dijalankan pada GPU (misalnya A100 di Modal).

## 1. Persiapan Lingkungan

Instal dependensi inti AutoGluon dan pustaka terkait. Gunakan perintah `pip` berikut (jalankan satu kali saja).

In [ ]:
%%capture
!pip install -U pip
!pip install -U autogluon "torch>=2.0.0" "torchvision>=0.15.0" pandas scikit-learn matplotlib seaborn


## 2. Import Pustaka dan Konfigurasi Awal

In [ ]:
import os
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from IPython.display import display

from autogluon.core import space
from autogluon.multimodal import MultiModalPredictor

import torch
from torchvision.datasets import CIFAR10

plt.style.use("seaborn-v0_8")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch version: {torch.__version__}")
print(f"CUDA tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU aktif: {torch.cuda.get_device_name(0)}")


## 3. Unduh dan Siapkan Dataset CIFAR-10

Blok kode berikut akan mengunduh CIFAR-10 dari sumber resmi, menyimpan setiap gambar sebagai berkas PNG, serta menyiapkan DataFrame dengan kolom `image` (path ke gambar) dan `label` yang dibutuhkan oleh AutoGluon.

In [ ]:
data_root = Path("data/cifar10")
image_root = data_root / "images"
train_image_root = image_root / "train"
test_image_root = image_root / "test"

for folder in [train_image_root, test_image_root]:
    folder.mkdir(parents=True, exist_ok=True)

print("Mengunduh dan mengekstrak CIFAR-10...")
train_dataset = CIFAR10(root=str(data_root), train=True, download=True)
test_dataset = CIFAR10(root=str(data_root), train=False, download=True)

print("Menyimpan gambar CIFAR-10 sebagai PNG...")

def export_split(dataset, split_root, split_name):
    records = []
    for idx, (img, label) in enumerate(dataset):
        class_name = dataset.classes[label]
        class_dir = split_root / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        file_path = class_dir / f"{split_name}_{idx:05d}.png"
        if not file_path.exists():
            img.save(file_path)
        records.append({"image": str(file_path.resolve()), "label": class_name})
    return pd.DataFrame(records)

train_full_df = export_split(train_dataset, train_image_root, "train")
test_df = export_split(test_dataset, test_image_root, "test")

print(f"Total data train: {len(train_full_df)}, data test: {len(test_df)}")

train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_full_df["label"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

def preview_df(df, name):
    print(f"Sample {name}:")
    display(df.sample(5, random_state=SEED))

preview_df(train_df, "train")
preview_df(val_df, "validation")
preview_df(test_df, "test")


## 4. Lihat Backbone yang Didukung AutoGluon

Gunakan `MultiModalPredictor.list_supported_models(pretrained=True)` untuk mengetahui backbone yang tersedia (pretrained) yang akan kita manfaatkan selama proses pelatihan dan HPO.

In [ ]:
supported_models = MultiModalPredictor.list_supported_models(pretrained=True)
print(f"Total backbone pretrained yang tersedia: {len(supported_models)}")
for model_name in supported_models:
    print(f"- {model_name}")


## 5. Latih Model dengan AutoGluon + HPO

Kita menggunakan seluruh backbone pretrained yang tersedia sekaligus menjalankan Hyperparameter Optimization (HPO) bawaan. Parameter pencarian dasar seperti *learning rate* diatur sebagai ruang pencarian log-uniform. Anda bisa menyesuaikan `hyperparameter_tune_kwargs` sesuai kapasitas GPU/komputasi.

In [ ]:
output_dir = Path("autogluon-artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

predictor = MultiModalPredictor(
    problem_type="multiclass",
    label="label",
    eval_metric="accuracy",
    path=str(output_dir),
)

hyperparameters = {
    "model.names": supported_models,
    "optimization.learning_rate": space.Real(1e-5, 1e-3, log=True),
    "optimization.max_epochs": 15,
    "optimization.batch_size": 64,
}

hyperparameter_tune_kwargs = {
    "num_trials": max(len(supported_models) * 2, 4),
    "scheduler": "local",
    "searcher": "bayesopt",
}

train_data = train_df.copy()
val_data = val_df.copy()

predictor.fit(
    train_data=train_data,
    tuning_data=val_data,
    time_limit=None,
    hyperparameters=hyperparameters,
    hyperparameter_tune_kwargs=hyperparameter_tune_kwargs,
    presets="best_quality",
)


## 6. Leaderboard Model

Periksa performa masing-masing model yang dihasilkan AutoGluon.

In [ ]:
leaderboard_df = predictor.leaderboard(val_df, extra_info=True)
display(leaderboard_df)


## 7. Analisis Feature Importance

AutoGluon menyediakan analisis *feature importance*. Untuk kasus ini, kita lakukan pada data validasi dan memvisualisasikan hasilnya.

In [ ]:
importance_df = predictor.feature_importance(val_df)
importance_df = importance_df.sort_values(by="importance", ascending=False)
display(importance_df)

plt.figure(figsize=(6, 4))
plt.title("Feature Importance (Validation)")
plt.barh(importance_df["feature"], importance_df["importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()


## 8. Evaluasi Model Terbaik

Gunakan data test untuk mengevaluasi model terbaik dan tinjau metrik yang dihasilkan.

In [ ]:
best_model = predictor.get_model_best()
print(f"Model terbaik: {best_model}")

test_metrics = predictor.evaluate(test_df)
print("\nEvaluasi pada data test:")
for metric, value in test_metrics.items():
    print(f"- {metric}: {value:.4f}")


## 9. Simpan Model

Simpan model terbaik beserta artefaknya untuk penggunaan lebih lanjut.

In [ ]:
model_export_path = output_dir / "cifar10_multimodal_predictor"
predictor.save(str(model_export_path))
print(f"Model disimpan di: {model_export_path}")


## 10. Prediksi Contoh (Opsional)

Anda dapat mencoba melakukan prediksi pada beberapa contoh dari data test.

In [ ]:
sample_df = test_df.sample(5, random_state=SEED).reset_index(drop=True)
display(sample_df)

predictions = predictor.predict(sample_df)
display(pd.DataFrame({"image": sample_df["image"], "ground_truth": sample_df["label"], "prediction": predictions}))
